# Tutorial 3: Gradient Descent from Scratch

**Prerequisites:** Tutorial 1 — What Is an Optimization Problem?, Tutorial 2 — Types of Optimization Problems  
**Up Next:** Tutorial 4 — Finite Differences and Numerical Gradients

---

## Concept

Gradient descent is the workhorse of continuous optimization. The idea is simple: at each step, move in the direction that decreases $f$ the fastest — the negative gradient. Repeating this produces a sequence of points that (hopefully) converges to a local minimum.

The algorithm has one critical hyper-parameter: the **step size** (or learning rate) $\alpha$. Too large and you overshoot; too small and convergence is glacially slow.

## Key Takeaway

> **Gradient descent iterates $\mathbf{x}_{k+1} = \mathbf{x}_k - \alpha \nabla f(\mathbf{x}_k)$.  It is simple and powerful, but the step size matters enormously and it can get stuck in local minima.**

## Math Core

$$\mathbf{x}_{k+1} = \mathbf{x}_k - \alpha \, \nabla f(\mathbf{x}_k)$$

where $\nabla f = \left[\frac{\partial f}{\partial x_1}, \frac{\partial f}{\partial x_2}\right]^T$ is the gradient. When we don't have an analytical gradient, we approximate it with **finite differences** (covered in depth in Tutorial 4):

$$\frac{\partial f}{\partial x_i} \approx \frac{f(\mathbf{x} + \delta \mathbf{e}_i) - f(\mathbf{x})}{\delta}$$

## Code

We reuse the four-bar objective from Tutorial 1, then implement gradient descent in under 15 lines.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dms.curves import CompareCurves
%matplotlib inline

# Constants (same as Tutorial 1)
L0, L1 = 1.0, 1.0
PX, PY = 0.5, 0.3
TARGETS = np.array([
    [1.0, 1.5], [0.75, 1.5], [0.5, 1.5],
    [0.25, 1.5], [0.0, 1.5], [-0.25, 1.5]
])
THETAS = np.linspace(np.deg2rad(60), np.deg2rad(120), len(TARGETS))

In [ ]:
def fourbar_fk(theta1, l0, l1, l2, l3):
    """Cosine-law closed-form FK for four-bar linkage."""
    ax, ay = l1 * np.cos(theta1), l1 * np.sin(theta1)
    dx, dy = ax - l0, ay
    d = np.sqrt(dx**2 + dy**2)
    cos_alpha = (l3**2 + d**2 - l2**2) / (2 * l3 * d)
    if np.abs(cos_alpha) > 1:
        return None
    alpha = np.arccos(np.clip(cos_alpha, -1, 1))
    beta = np.arctan2(dy, dx)
    theta3 = beta + alpha
    bx = l0 + l3 * np.cos(theta3)
    by = l3 * np.sin(theta3)
    theta2 = np.arctan2(by - ay, bx - ax)
    return theta2, theta3


def coupler_point(theta1, l2, l3):
    """Compute coupler point for given crank angle."""
    sol = fourbar_fk(theta1, L0, L1, l2, l3)
    if sol is None:
        return None
    theta2, _ = sol
    ax, ay = L1 * np.cos(theta1), L1 * np.sin(theta1)
    ux, uy = np.cos(theta2), np.sin(theta2)
    return np.array([ax + PX * ux - PY * uy,
                     ay + PX * uy + PY * ux])


def objective(x):
    """Path-synthesis objective."""
    l2, l3 = x
    path = []
    for theta1 in THETAS:
        pt = coupler_point(theta1, l2, l3)
        if pt is None:
            return 1e3
        path.append(pt)
    path = np.array(path)
    return CompareCurves(path[:, 0], path[:, 1],
                         TARGETS[:, 0], TARGETS[:, 1])

### Gradient descent implementation

We approximate the gradient with forward finite differences and iterate.

In [ ]:
def gradient_descent(f, x0, alpha=0.05, delta=1e-5, n_iter=200,
                     bounds=(0.3, 2.5)):
    """Gradient descent with forward-difference gradients."""
    x = np.array(x0, dtype=float)
    history = [x.copy()]
    for _ in range(n_iter):
        fx = f(x)
        grad = np.zeros_like(x)
        for i in range(len(x)):
            x_fwd = x.copy()
            x_fwd[i] += delta
            grad[i] = (f(x_fwd) - fx) / delta
        x = x - alpha * grad
        x = np.clip(x, bounds[0], bounds[1])
        history.append(x.copy())
    return np.array(history)

In [ ]:
# Run gradient descent from a starting guess
x0 = [1.8, 1.8]
history = gradient_descent(objective, x0, alpha=0.05, n_iter=150)

costs = [objective(h) for h in history]
print(f'Start: x={history[0]}, f={costs[0]:.4f}')
print(f'End:   x={history[-1].round(4)}, f={costs[-1]:.4f}')

### Convergence curve

Plotting $f$ versus iteration number shows how quickly the algorithm improves.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.semilogy(costs, 'b-', lw=2)
ax.set_xlabel('Iteration')
ax.set_ylabel(r'$f(\mathbf{x})$')
ax.set_title('Gradient descent convergence')
ax.grid(True, alpha=0.3)
plt.tight_layout()

### Trajectory on the objective landscape

Let's overlay the gradient descent path on the contour plot to see how it navigates the landscape.

In [ ]:
# Build contour grid
l2_vals = np.linspace(0.3, 2.5, 80)
l3_vals = np.linspace(0.3, 2.5, 80)
L2, L3 = np.meshgrid(l2_vals, l3_vals)
Z = np.vectorize(lambda a, b: objective([a, b]))(L2, L3)

fig, ax = plt.subplots(figsize=(7, 5))
cf = ax.contourf(L2, L3, np.log1p(Z), levels=30, cmap='viridis')
ax.plot(history[:, 0], history[:, 1], 'r.-', ms=3, lw=1,
        label='GD trajectory')
ax.plot(*history[0], 'wo', ms=8, label='start')
ax.plot(*history[-1], 'r*', ms=12, label='end')
ax.set_xlabel(r'$\ell_2$')
ax.set_ylabel(r'$\ell_3$')
ax.set_title('Gradient descent trajectory')
ax.legend()
plt.colorbar(cf, label=r'$\ln(1 + f)$')
plt.tight_layout()

### Step size matters

Let's compare three different learning rates to see the effect.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for alpha, color in [(0.01, 'blue'), (0.05, 'green'), (0.2, 'red')]:
    h = gradient_descent(objective, x0, alpha=alpha, n_iter=150)
    c = [objective(pt) for pt in h]
    ax.semilogy(c, color=color, lw=2, label=f'$\\alpha$={alpha}')
ax.set_xlabel('Iteration')
ax.set_ylabel(r'$f(\mathbf{x})$')
ax.set_title('Effect of step size on convergence')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

### Local minima: starting point matters

Since our problem is non-convex (Tutorial 2), different starting points can lead to different local minima.

In [ ]:
starts = [[0.8, 1.0], [1.8, 1.8], [2.0, 1.0]]
colors = ['cyan', 'red', 'yellow']

fig, ax = plt.subplots(figsize=(7, 5))
cf = ax.contourf(L2, L3, np.log1p(Z), levels=30, cmap='viridis')
for x0s, col in zip(starts, colors):
    h = gradient_descent(objective, x0s, alpha=0.05, n_iter=200)
    ax.plot(h[:, 0], h[:, 1], '.-', color=col, ms=2, lw=1)
    ax.plot(*h[0], 'o', color=col, ms=8, markeredgecolor='k')
    ax.plot(*h[-1], '*', color=col, ms=12, markeredgecolor='k')
ax.set_xlabel(r'$\ell_2$')
ax.set_ylabel(r'$\ell_3$')
ax.set_title('Different starts $\\to$ different local minima')
plt.colorbar(cf, label=r'$\ln(1 + f)$')
plt.tight_layout()

## Mechanism Hook

Let's see the mechanism at the solution gradient descent found.

In [ ]:
from dms.mechanisms.fourbar import FourBar

x_best = history[-1]
l2, l3 = x_best
mechanism = FourBar(L0, L1, l2, l3)
pts = [coupler_point(t, l2, l3) for t in THETAS]
path = np.array([p for p in pts if p is not None])

fig, ax = plt.subplots(figsize=(6, 5))
mechanism.plot(np.deg2rad(90), ax=ax)
ax.plot(TARGETS[:, 0], TARGETS[:, 1], 'r*', ms=10, label='targets')
if len(path) > 0:
    ax.plot(path[:, 0], path[:, 1], 'b.-', label='coupler path')
ax.set_aspect('equal')
ax.legend()
ax.set_title(f'GD result: $\\ell_2$={l2:.3f}, $\\ell_3$={l3:.3f}'
             f'  |  f = {objective(x_best):.4f}')
plt.tight_layout()

Gradient descent got us to a reasonable solution, but we relied on a crude finite-difference gradient. Tutorial 4 digs into how to compute these numerical gradients accurately and efficiently.

---